# MTA Daily Ridership – Data Preparation
**DEPI · Professional Data Analysis · Team 1**

This notebook implements the data-quality methodology from the project documentation (steps 1–6) and
prepares the data for the database design (`date_dim`, `service_dim`, `ridership`).

| Step | What it does |
|---|---|
| 0 | Setup |
| 1 | Load the raw CSV |
| 2 | Parse `Date` and convert numeric fields |
| 3 | Check duplicate dates and date gaps |
| 4 | Profile missing values |
| 5 | Inspect unusual values and sudden changes |
| 6 | Reshape to the ER design, validate, and save outputs + log |

**Rule:** missing values are never silently replaced by zero.

## Step 0 – Setup

In [35]:
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
from pathlib import Path

path=r'D:\DEPI\final\MTA Daily Ridership\MTA_Daily_Ridership.csv'
DATA_PATH = Path(path)
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

# Transformation log: every check/action is recorded here and saved at the end
log_rows = []

def log(step, check, result, action):
    log_rows.append({"step": step, "check": check, "result": result, "action": action})
    print(f"[Step {step}] {check}: {result}  ->  {action}")

## Step 1 – Load the raw data

In [19]:
raw = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", raw.shape)
print()
print(raw.dtypes.to_string())

Shape (rows, columns): (1706, 15)

Date                                                         str
Subways: Total Estimated Ridership                         int64
Subways: % of Comparable Pre-Pandemic Day                  int64
Buses: Total Estimated Ridership                           int64
Buses: % of Comparable Pre-Pandemic Day                    int64
LIRR: Total Estimated Ridership                            int64
LIRR: % of Comparable Pre-Pandemic Day                     int64
Metro-North: Total Estimated Ridership                     int64
Metro-North: % of Comparable Pre-Pandemic Day              int64
Access-A-Ride: Total Scheduled Trips                       int64
Access-A-Ride: % of Comparable Pre-Pandemic Day            int64
Bridges and Tunnels: Total Traffic                         int64
Bridges and Tunnels: % of Comparable Pre-Pandemic Day      int64
Staten Island Railway: Total Estimated Ridership           int64
Staten Island Railway: % of Comparable Pre-Pandemic Day

In [20]:
raw.head()

,Date,Subways: Total Estimated Ridership,Subways: % of Comparable Pre-Pandemic Day,Buses: Total Estimated Ridership,Buses: % of Comparable Pre-Pandemic Day,LIRR: Total Estimated Ridership,LIRR: % of Comparable Pre-Pandemic Day,Metro-North: Total Estimated Ridership,Metro-North: % of Comparable Pre-Pandemic Day,Access-A-Ride: Total Scheduled Trips,Access-A-Ride: % of Comparable Pre-Pandemic Day,Bridges and Tunnels: Total Traffic,Bridges and Tunnels: % of Comparable Pre-Pandemic Day,Staten Island Railway: Total Estimated Ridership,Staten Island Railway: % of Comparable Pre-Pandemic Day
0,2020-03-01,2212965,97,984908,99,86790,100,55825,59,19922,113,786960,98,1636,52
1,2020-03-02,5329915,96,2209066,99,321569,103,180701,66,30338,102,874619,95,17140,107
2,2020-03-03,5481103,98,2228608,99,319727,102,190648,69,32767,110,882175,96,17453,109
3,2020-03-04,5498809,99,2177165,97,311662,99,192689,70,34297,115,905558,98,17136,107
4,2020-03-05,5496453,99,2244515,100,307597,98,194386,70,33209,112,929298,101,17203,108


## Step 2 – Parse `Date` and convert numeric fields safely

In [21]:
df = raw.copy()
df["Date"] = pd.to_datetime(df["Date"], errors="raise")

value_cols = [c for c in df.columns if c != "Date"]
na_before = int(df[value_cols].isna().sum().sum())
df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
new_na = int(df[value_cols].isna().sum().sum()) - na_before

log("2", "Date parsed to datetime", "OK", "converted")
log("2", "values that failed numeric conversion", new_na,
    "none" if new_na == 0 else "review these cells")
print()
print(df.dtypes.value_counts().to_string())

[Step 2] Date parsed to datetime: OK  ->  converted
[Step 2] values that failed numeric conversion: 0  ->  none

int64             14
datetime64[us]     1


## Step 3 – Duplicate dates and date gaps

In [22]:
dup_dates = int(df["Date"].duplicated().sum())
full_range = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
missing_days = full_range.difference(df["Date"])

log("3", "duplicate dates", dup_dates, "nothing to remove" if dup_dates == 0 else "review duplicates")
log("3", "missing calendar days", len(missing_days), "none" if len(missing_days) == 0 else "list the gaps")
print(f"Coverage: {df['Date'].min().date()} -> {df['Date'].max().date()}  ({len(df)} rows)")

[Step 3] duplicate dates: 0  ->  nothing to remove
[Step 3] missing calendar days: 0  ->  none
Coverage: 2020-03-01 -> 2024-10-31  (1706 rows)


## Step 4 – Missing values by column

In [23]:
missing = df[value_cols].isna().sum()
missing_report = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})

log("4", "total missing values", int(missing.sum()),
    "kept as NULL (never replaced by 0)")
missing_report[missing_report["missing"] > 0] if missing.sum() else print("No missing values in any column.")

[Step 4] total missing values: 0  ->  kept as NULL (never replaced by 0)
No missing values in any column.


## Step 5 – Unusual values and sudden changes
Large day-to-day changes are expected because weekdays and weekends differ.
So changes are measured against the **same weekday one week earlier**.

In [24]:
total_cols = [c for c in value_cols if "%" not in c]
pct_cols = [c for c in value_cols if "%" in c]

negatives = int((df[value_cols] < 0).sum().sum())
log("5", "negative values", negatives, "none" if negatives == 0 else "investigate")

zero_counts = (df[value_cols] == 0).sum()
zero_counts = zero_counts[zero_counts > 0]
log("5", "columns that contain zeros", len(zero_counts), "inspect below before deciding")
zero_counts

[Step 5] negative values: 0  ->  none
[Step 5] columns that contain zeros: 2  ->  inspect below before deciding


Staten Island Railway: Total Estimated Ridership           32
Staten Island Railway: % of Comparable Pre-Pandemic Day    54
dtype: int64

In [25]:
# Week-over-week change (same weekday). inf appears when the previous value is 0.
wow = df.set_index("Date")[total_cols].pct_change(7)
big_changes = (wow.abs() > 1).sum().rename("days with change > 100%")
big_changes

Subways: Total Estimated Ridership                   9
Buses: Total Estimated Ridership                    21
LIRR: Total Estimated Ridership                     21
Metro-North: Total Estimated Ridership              24
Access-A-Ride: Total Scheduled Trips                22
Bridges and Tunnels: Total Traffic                   4
Staten Island Railway: Total Estimated Ridership    74
Name: days with change > 100%, dtype: int64

In [26]:
# Staten Island Railway: when are the zero-ridership days?
sir = "Staten Island Railway: Total Estimated Ridership"
zero_days = df.loc[df[sir] == 0, "Date"]

if len(zero_days):
    run_id = zero_days.diff().dt.days.ne(1).cumsum()
    zero_runs = zero_days.groupby(run_id).agg(start="min", end="max", days="count")
    display(zero_runs)
else:
    print("No zero days.")

log("5", "Staten Island Railway zero-ridership days", int(len(zero_days)),
    "kept as reported; confirm against the data dictionary (do not treat as missing)")

,start,end,days
Date,,,
1,2020-06-21,2020-06-21,1
2,2020-10-24,2020-10-25,2
3,2021-03-13,2021-03-14,2
4,2021-03-20,2021-03-21,2
5,2021-03-27,2021-03-28,2
6,2021-04-17,2021-04-18,2
7,2021-04-24,2021-04-25,2
8,2021-05-01,2021-05-02,2
9,2021-05-08,2021-05-08,1


[Step 5] Staten Island Railway zero-ridership days: 32  ->  kept as reported; confirm against the data dictionary (do not treat as missing)


## Step 6 – Reshape to the ER design and validate
The wide CSV (15 columns) becomes three tables:
`date_dim`, `service_dim`, and `ridership` (long format).

In [27]:
# 6a. wide -> long
long = df.melt(id_vars="Date", var_name="column", value_name="value")
parts = long["column"].str.split(": ", n=1, expand=True)
long["service_name"] = parts[0]
long["kind"] = np.where(parts[1].str.startswith("%"), "pct_pre_pandemic", "total")

# 6b. service dimension (keeps the original column order)
measure_map = {"Access-A-Ride": "Scheduled trips", "Bridges and Tunnels": "Traffic"}
services = list(parts[0].unique())
service_dim = pd.DataFrame({"service_id": range(1, len(services) + 1), "service_name": services})
service_dim["measure_type"] = service_dim["service_name"].map(measure_map).fillna("Estimated ridership")

# 6c. fact table
ridership = (long.pivot(index=["Date", "service_name"], columns="kind", values="value")
                 .reset_index()
                 .rename(columns={"Date": "date"}))
ridership.columns.name = None
ridership = (ridership.merge(service_dim[["service_id", "service_name"]], on="service_name")
                      [["date", "service_id", "total", "pct_pre_pandemic"]]
                      .sort_values(["date", "service_id"]).reset_index(drop=True))

# 6d. date dimension (holidays are not flagged: they are not in the source file)
date_dim = pd.DataFrame({"date": df["Date"].sort_values().values})
dow = date_dim["date"].dt.dayofweek
date_dim["day_type"] = np.where(dow == 5, "Saturday", np.where(dow == 6, "Sunday", "Weekday"))
date_dim["month"] = date_dim["date"].dt.month
date_dim["year"] = date_dim["date"].dt.year

print("ridership:", ridership.shape, "| service_dim:", service_dim.shape, "| date_dim:", date_dim.shape)
service_dim

ridership: (11942, 4) | service_dim: (7, 3) | date_dim: (1706, 4)


,service_id,service_name,measure_type
0,1,Subways,Estimated ridership
1,2,Buses,Estimated ridership
2,3,LIRR,Estimated ridership
3,4,Metro-North,Estimated ridership
4,5,Access-A-Ride,Scheduled trips
5,6,Bridges and Tunnels,Traffic
6,7,Staten Island Railway,Estimated ridership


In [28]:
# 6e. validation checks (the notebook stops with an error if any check fails)
assert len(ridership) == len(df) * len(service_dim), "row count mismatch"
assert not ridership.duplicated(["date", "service_id"]).any(), "duplicate (date, service) keys"
assert ridership[["total", "pct_pre_pandemic"]].notna().all().all(), "unexpected missing values"

orig = long[long["kind"] == "total"].groupby("service_name")["value"].sum()
new = (ridership.merge(service_dim, on="service_id")
                .groupby("service_name")["total"].sum())
assert (orig.sort_index() == new.sort_index()).all(), "totals changed during reshaping"

last = (ridership[ridership["date"] == ridership["date"].max()]
        .merge(service_dim, on="service_id").set_index("service_name"))
assert last.loc["Subways", "pct_pre_pandemic"] == 70
assert last.loc["Buses", "pct_pre_pandemic"] == 58

log("6", "expected rows = days x services", f"{len(ridership):,}", "validated")
log("6", "totals reconcile with source file", "match", "validated")
log("6", "latest date check (Subways 70%, Buses 58%)", str(ridership['date'].max().date()), "validated")
print("All validation checks passed.")

[Step 6] expected rows = days x services: 11,942  ->  validated
[Step 6] totals reconcile with source file: match  ->  validated
[Step 6] latest date check (Subways 70%, Buses 58%): 2024-10-31  ->  validated
All validation checks passed.


## Save outputs (CSV files + SQL Server + cleaning log)
**Before running the SQL Server cells:** run the install cell (`sqlalchemy` + `pyodbc`), restart the kernel once,
and make sure the *ODBC Driver for SQL Server* (17 or 18) is installed. Then set `SERVER` below to your instance name.
If the connection fails, the notebook keeps going and records the problem in the log.

In [29]:
ridership.to_csv(OUT_DIR / "ridership.csv", index=False)
service_dim.to_csv(OUT_DIR / "service_dim.csv", index=False)
date_dim.to_csv(OUT_DIR / "date_dim.csv", index=False)
log("6", "CSV files saved", str(OUT_DIR), "ridership.csv, service_dim.csv, date_dim.csv")

[Step 6] CSV files saved: output  ->  ridership.csv, service_dim.csv, date_dim.csv


In [31]:
# ---- SQL Server connection settings (edit these) ----
SERVER = r"localhost"
DATABASE = "MTA_Ridership"
DRIVER = "ODBC Driver 18 for SQL Server"

def odbc_url(database):
    conn = (f"DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={database};"
            "Trusted_Connection=yes;TrustServerCertificate=yes")
    return "mssql+pyodbc:///?odbc_connect=" + quote_plus(conn)

try:
    import pyodbc
    print("ODBC drivers found on this PC:", [d for d in pyodbc.drivers() if "SQL Server" in d])
except ImportError:
    print("pyodbc is not installed yet - run the cell above, then restart the kernel.")

ODBC drivers found on this PC: ['SQL Server', 'ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server']


In [32]:
sql_ok = False
try:
    from sqlalchemy import create_engine

    # 1) create the database if needed
    master = create_engine(odbc_url("master"), isolation_level="AUTOCOMMIT")
    with master.connect() as con:
        con.exec_driver_sql(f"IF DB_ID('{DATABASE}') IS NULL CREATE DATABASE [{DATABASE}]")

    # 2) create the tables (physical schema from the design document)
    engine = create_engine(odbc_url(DATABASE), fast_executemany=True)
    ddl = [
        "IF OBJECT_ID('dbo.ridership', 'U') IS NOT NULL DROP TABLE dbo.ridership",
        "IF OBJECT_ID('dbo.service_dim', 'U') IS NOT NULL DROP TABLE dbo.service_dim",
        "IF OBJECT_ID('dbo.date_dim', 'U') IS NOT NULL DROP TABLE dbo.date_dim",
        "CREATE TABLE dbo.date_dim ([date] DATE NOT NULL PRIMARY KEY, day_type NVARCHAR(10) NOT NULL, [month] TINYINT NOT NULL, [year] SMALLINT NOT NULL)",
        "CREATE TABLE dbo.service_dim (service_id SMALLINT NOT NULL PRIMARY KEY, service_name NVARCHAR(50) NOT NULL, measure_type NVARCHAR(30) NOT NULL)",
        "CREATE TABLE dbo.ridership ([date] DATE NOT NULL, service_id SMALLINT NOT NULL, total INT NULL, pct_pre_pandemic SMALLINT NULL, "
        "CONSTRAINT PK_ridership PRIMARY KEY ([date], service_id), "
        "CONSTRAINT FK_ridership_date FOREIGN KEY ([date]) REFERENCES dbo.date_dim([date]), "
        "CONSTRAINT FK_ridership_service FOREIGN KEY (service_id) REFERENCES dbo.service_dim(service_id))",
    ]
    with engine.begin() as con:
        for stmt in ddl:
            con.exec_driver_sql(stmt)

    # 3) load the data (dimensions first, then the fact table)
    date_dim.assign(date=date_dim["date"].dt.date).to_sql("date_dim", engine, schema="dbo", if_exists="append", index=False)
    service_dim.to_sql("service_dim", engine, schema="dbo", if_exists="append", index=False)
    ridership.assign(date=ridership["date"].dt.date).to_sql("ridership", engine, schema="dbo", if_exists="append", index=False, chunksize=5000)

    sql_ok = True
    log("6", "SQL Server load", f"{SERVER} / {DATABASE}", "date_dim, service_dim, ridership loaded")
except Exception as e:
    log("6", "SQL Server load", "FAILED", "see the full error below")
    print("\nFULL ERROR:\n", type(e).__name__, "-", e)

[Step 6] SQL Server load: localhost / MTA_Ridership  ->  date_dim, service_dim, ridership loaded


In [33]:
# Quick T-SQL check: average % of pre-pandemic level per service in the last 12 months
query = """
SELECT s.service_name,
       ROUND(AVG(CAST(r.pct_pre_pandemic AS FLOAT)), 1) AS avg_pct_pre_pandemic
FROM dbo.ridership r
JOIN dbo.service_dim s ON s.service_id = r.service_id
WHERE r.[date] >= '2023-11-01'
GROUP BY s.service_name
ORDER BY avg_pct_pre_pandemic DESC;
"""
if sql_ok:
    display(pd.read_sql(query, engine))
else:
    print("Skipped: the SQL Server load did not run.")

,service_name,avg_pct_pre_pandemic
0,Access-A-Ride,117.5
1,Bridges and Tunnels,101.1
2,LIRR,87.5
3,Metro-North,78.0
4,Subways,71.3
5,Buses,59.9
6,Staten Island Railway,50.9


## Transformation log

In [34]:
cleaning_log = pd.DataFrame(log_rows)
cleaning_log.to_csv(OUT_DIR / "cleaning_log.csv", index=False)
cleaning_log

,step,check,result,action
0,2,Date parsed to datetime,OK,converted
1,2,values that failed numeric conversion,0,none
2,3,duplicate dates,0,nothing to remove
3,3,missing calendar days,0,none
4,4,total missing values,0,kept as NULL (never replaced by 0)
5,5,negative values,0,none
6,5,columns that contain zeros,2,inspect below before deciding
7,5,Staten Island Railway zero-ridership days,32,kept as reported; confirm against the data dic...
8,6,expected rows = days x services,"11,942",validated
9,6,totals reconcile with source file,match,validated
